# SafeReport Crew

**A multi-agent copilot that helps an OSHA area office decide, for each severe injury report, whether
to send an inspector or a letter.**

IE5374 Applied Generative AI, final project of Seven Team-5374: Quoc Hung Le (Kevin), Hamad Al
Sulaimi, Khoa Tran. Code: [github.com/lequo-neu/safereport-crew](https://github.com/lequo-neu/safereport-crew).

**How to run this notebook.** Choose *Runtime > Run all*. One cell asks you to upload OSHA's
public data file (step 6 explains how to get it). Everything else runs by itself.

**Cost and keys.** The notebook is offline. No API key is read or needed, and no model is
called: every model step is answered by the project's fixed offline rules, and the notebook
checks this before it runs the crew. Nothing costs money.

| Part | What it shows | Owner |
|---|---|---|
| 1 to 8 | the problem, the seven agents, install, OSHA data, building the data and indexes | Hamad |
| 9 to 16 | one real week of Texas reports run through the seven agents, step by step | Hamad |
| 17 | the model router: accuracy per model and step, and why each step goes where it goes | Khoa |
| 18 | the evaluation numbers, read from `eval/results/` | Khoa |
| 19 | edge cases the system handles | Khoa |
| 20 | summary | together |

## 1. The problem

When a US worker is hospitalized, loses a limb or loses an eye at work, the employer must report it
to OSHA within 24 hours. Each report is a short free-text story ("an employee's hand was pulled
into the rollers while clearing a jam"). For every report, an OSHA area office chooses between:

* an **on-site inspection**: an inspector visits. Inspector days are scarce; or
* an **RRI letter** (Rapid Response Investigation): the employer investigates and replies within
  five working days.

Some reports *must* be inspected under OSHA's **Category 1** rules (for example two or more workers
hospitalized, or a hazard covered by a National Emphasis Program). The rest compete for the
inspector days left. Today staff read every story, look up the employer's history and search the
regulations by hand.

SafeReport Crew does that reading, checking and planning, then hands a **proposed** list to a
person, who approves it or asks for changes. Nothing is sent without that approval.

## 2. The seven agents, and why each one exists

The crew is a [LangGraph](https://langchain-ai.github.io/langgraph/) graph. Each agent does one job
that a person in the area office does today, and passes a structured result to the next one.

| Agent | Kind | What it does | Why it is a separate agent |
|---|---|---|---|
| **Planner** | reasoning | splits the request into tasks and asks for what is missing (for example the state) | a vague request must not be guessed; asking is cheaper than a wrong week |
| **Intake** | reasoning | reads each story: activity, hazard, injury, an OSHA event code with a confidence, and the facts the story leaves out | every later step depends on what happened, read the same way each time |
| **Standards** | retrieval | searches the federal regulations (29 CFR 1910 from eCFR) and cites the section and paragraph | an inspection or letter must name the rule; it comes from the official text, not from memory |
| **Analyst** | data | the employer's past reports, similar past injuries, how common the hazard is in the industry | a repeat report from the same employer within a year is a Category 1 reason to inspect |
| **Optimizer** | optimization | applies Category 1 first, then an integer program (PuLP) spends the inspector days left | choosing which reports get the scarce days is a budget problem, solved exactly |
| **Writer** | generation | the summary and the RRI letters; every number is a placeholder filled from a ledger of recorded facts | text is where a model would invent numbers; here it cannot |
| **Decision critic** | checking | checks each citation against eCFR and OSHA's own rules, each number and the plan rules, and sends work back | a second, independent check before a person sees the list |

**Two stages.** Stage 1 codes every report of the week with fast rules and a trained classifier,
without a model. Only reports that are unclear, severe or close to the cut go to stage 2, where
Intake, Standards and Analyst run on each of them in parallel branches. Part 12 counts how many
went each way in the week shown here, and compares the count with the B4 evaluation.

## 3. Settings

The only cell you may want to edit.

* `REPO_REF` pins the exact version of the code, so the notebook gives the same results every
  time. It is a commit of `main`; it becomes the release tag once the team tags the hand-in
  version.
* `USE_EMBEDDINGS = False` builds a keyword (BM25) search only, which is quick on a free Colab
  CPU. `True` adds a small embedding model (BAAI/bge-small-en-v1.5) to the regulation search and
  the similar injury search. It needs a few hundred MB and takes much longer on a CPU; a GPU
  runtime helps. The results then differ from the BM25 run.
* `DATA_SOURCE` is `"upload"` (a file picker) or `"drive"` (a file in your Google Drive).

In [ ]:
REPO_URL = "https://github.com/lequo-neu/safereport-crew.git"
# The commit of main on 24 November 2026; replace it with the release tag once the team tags one.
REPO_REF = "fc755be90fbafbb8ec0333741d2279d459ca2ac6"
USE_EMBEDDINGS = False
DATA_SOURCE = "upload"  # "upload" or "drive"
DRIVE_FILE = "/content/drive/MyDrive/January2015toNovember2025.zip"  # used when DATA_SOURCE == "drive"

## 4. Check the runtime

The project supports Python 3.11 to 3.13. This cell also removes any model API key variables from
this session, so no step below can reach a paid model even by mistake. It prints only how many it
removed, never a value.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys


IN_COLAB = "google.colab" in sys.modules  # Colab loads it when the runtime starts
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"SafeReport needs Python 3.11 to 3.13; this runtime has {sys.version.split()[0]}")

# On a local machine (used by the team's tests), these environment variables point at a folder.
WORK_DIR = (
    Path("/content") if IN_COLAB else Path(os.environ.get("SAFEREPORT_NOTEBOOK_WORKDIR", "notebook_work"))
)
WORK_DIR = WORK_DIR.resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get("SAFEREPORT_NOTEBOOK_REPO_URL", REPO_URL)
REPO_DIR = WORK_DIR / "safereport-crew"

KEY_VARIABLES = [
    "ANTHROPIC_API_KEY",
    "ANTHROPIC_AUTH_TOKEN",
    "OPENAI_API_KEY",
    "GOOGLE_API_KEY",
    "GEMINI_API_KEY",
]
removed = [name for name in KEY_VARIABLES if os.environ.pop(name, None) is not None]
print(f"Python {sys.version.split()[0]}, {'Google Colab' if IN_COLAB else 'local machine'}")
print(f"Working folder: {WORK_DIR}")
print(f"Model API key variables removed from this session: {len(removed)}")

## 5. Install SafeReport Crew at the pinned version

This clones the public repository and checks out `REPO_REF`. On Colab it then installs the package
and its dependencies with pip (a few minutes). The `models` extra, which holds the paid model
clients, is **not** installed.

**An expected pip message on Colab.** SafeReport needs pandas 3, and Colab's own `google-colab`
package pins pandas 2.2.3, so pip prints a red `ERROR: pip's dependency resolver ...` line saying
`google-colab ... requires pandas==2.2.3, but you have pandas 3.x`. The install itself succeeds
(the cell stops if pip fails). It is a real conflict, not hidden here: Colab's own pandas-based
extras (interactive tables, chart suggestions) were built for pandas 2.2.3 and may not work in
this session. This notebook does not use them; it shows tables with IPython's plain display.

If pandas was already loaded in this session before the install, the cell stops and asks you to
choose *Runtime > Restart session* and run all again (the clone and install are then skipped), so
the notebook never runs on two pandas versions at once.

In [ ]:
def run(*command: str, cwd: Path | None = None) -> None:
    """Run a command, print its output as it comes, and stop the notebook if it fails."""
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=child_env()
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"failed with exit code {process.returncode}: {' '.join(command)}")


def child_env() -> dict[str, str]:
    """The environment for commands: no API keys, and the pinned source first on the path."""
    env = {k: v for k, v in os.environ.items() if k not in KEY_VARIABLES}
    env["PYTHONPATH"] = os.pathsep.join(filter(None, [str(REPO_DIR / "src"), env.get("PYTHONPATH", "")]))
    return env


if not (REPO_DIR / ".git").exists():
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
run("git", "-c", "advice.detachedHead=false", "checkout", "--quiet", "--detach", REPO_REF, cwd=REPO_DIR)
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()

extras = "[embeddings]" if USE_EMBEDDINGS else ""
marker = REPO_DIR / f".notebook_installed_{COMMIT[:12]}{extras}"
if IN_COLAB and not marker.exists():
    run(sys.executable, "-m", "pip", "install", "--quiet", "-e", f"{REPO_DIR}{extras}")
    marker.touch()
elif not IN_COLAB:
    print("Local machine: the dependencies of the current environment are used; nothing is installed.")

from importlib.metadata import version


pandas_installed = version("pandas")
pandas_loaded = getattr(sys.modules.get("pandas"), "__version__", None)
if pandas_loaded not in (None, pandas_installed):
    raise RuntimeError(
        f"pandas {pandas_loaded} is loaded in this session but pandas {pandas_installed} is now installed. "
        "Choose Runtime > Restart session, then Run all again (the install is skipped)."
    )
print(f"pandas {pandas_installed} installed" + (", not yet loaded" if pandas_loaded is None else ", loaded"))

# The kernel was started before the install, so put the pinned source first on its path.
sys.path.insert(0, str(REPO_DIR / "src"))
import safereport


print(f"SafeReport {safereport.__version__} at commit {COMMIT[:12]} from {Path(safereport.__file__).parent}")
assert Path(safereport.__file__).resolve().is_relative_to(REPO_DIR.resolve()), (
    "safereport is not the pinned copy"
)

## 6. Load OSHA's severe injury reports

The data is OSHA's public **Severe Injury Reports** file. The pinned code, its 2025 holdout and
every result in `eval/results/` were built from the release that ends in **November 2025**
(`January2015toNovember2025.zip`), so this notebook uses that release and no other.

**OSHA's page now links a newer release.** Since November 2026, *Download the full SIR data set* on
<https://www.osha.gov/severe-injury-reports> gives `January2015toDecember2025.zip`. Do not use it
here: it adds December 2025 reports, which would change the data the results were measured on.
OSHA still serves the November release at its own address; open it in a browser (OSHA blocks
scripted downloads):

<https://www.osha.gov/sites/default/files/January2015toNovember2025.zip> (16,224,511 bytes, checked on
24 November 2026)

Then run the next cell and choose that file in the picker, or set `DATA_SOURCE = "drive"` and put
it at `DRIVE_FILE` in your Google Drive. The zip or the CSV inside it are both accepted. The cell
refuses any other release by its name and by its last report date, checks the columns, and skips
the upload when the file is already in place.

In [ ]:
import csv
import hashlib
import re
import shutil
import zipfile

import pandas as pd

from safereport.data import COLUMNS  # the columns the project reads from OSHA's file


CSV_NAME = "January2015toNovember2025.csv"
NOVEMBER_URL = "https://www.osha.gov/sites/default/files/January2015toNovember2025.zip"
LAST_DAY = "2025-11-30"  # the last report date of the November 2025 release
# The zip the project's data was built from; a different hash is reported, not refused.
NOVEMBER_ZIP_SHA256 = "a3f7f434e200fb956131f12277378e592993a25db3f328716fbece106f846bb0"
RAW_DIR = REPO_DIR / "data" / "raw"
RAW_CSV = RAW_DIR / CSV_NAME
RELEASE = re.compile(r"January2015to([A-Za-z]+)(\d{4})", re.IGNORECASE)


class WrongReleaseError(ValueError):
    """OSHA's file, but not the November 2025 release the project is pinned to."""


def check_release(name: str) -> None:
    found = RELEASE.search(name)
    if found and f"{found.group(1).title()}{found.group(2)}" != "November2025":
        raise WrongReleaseError(
            f"{name} is OSHA's release ending in {found.group(1).title()} {found.group(2)}, not the November "
            f"2025 release this notebook and its results are pinned to. OSHA's page now links the newer "
            f"release by default; download the November one from {NOVEMBER_URL} and run this cell again."
        )


def install_osha_file(source: Path) -> None:
    """Copy the CSV, or extract it from OSHA's zip, into data/raw of the pinned copy."""
    check_release(source.name)
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256(source.read_bytes()).hexdigest()
    print(f"{source.name}: {source.stat().st_size:,} bytes, sha256 {digest[:16]}...")
    if zipfile.is_zipfile(source):
        if digest != NOVEMBER_ZIP_SHA256:
            print(
                "Note: this zip is not byte for byte the one the project was built from; results may differ."
            )
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                check_release(Path(member.filename).name)
            members = [m for m in archive.infolist() if Path(m.filename).name == CSV_NAME]
            if len(members) != 1:
                raise ValueError(f"the zip does not hold {CSV_NAME}; download it from {NOVEMBER_URL}")
            if members[0].file_size > 500e6:
                raise ValueError("the CSV in the zip is unexpectedly large; refusing to extract it")
            with archive.open(members[0]) as src, RAW_CSV.open("wb") as dst:
                shutil.copyfileobj(src, dst)
    elif source.suffix.lower() == ".csv":
        shutil.copyfile(source, RAW_CSV)
    else:
        raise ValueError(f"expected OSHA's .zip or the .csv inside it, from {NOVEMBER_URL}")


def find_osha_file() -> Path:
    if not IN_COLAB:
        return Path(os.environ["SAFEREPORT_NOTEBOOK_DATA"])
    if DATA_SOURCE == "drive":
        from google.colab import drive

        drive.mount("/content/drive")
        return Path(DRIVE_FILE)
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("upload exactly one file: OSHA's zip or its CSV")
    name, content = next(iter(uploaded.items()))
    path = WORK_DIR / "upload" / Path(name).name
    path.parent.mkdir(exist_ok=True)
    path.write_bytes(content)
    return path


if RAW_CSV.exists():
    print(f"{CSV_NAME} is already in place; skipping the upload.")
else:
    install_osha_file(find_osha_file())

with RAW_CSV.open(encoding="utf-8-sig", errors="replace", newline="") as handle:
    header = next(csv.reader(handle))
missing = set(COLUMNS) - {name.strip() for name in header}
if missing:
    RAW_CSV.unlink()
    raise ValueError(f"this is not OSHA's SIR file; missing columns {sorted(missing)}. Run the cell again.")
# A renamed file is caught here: the November release has no report after its last day.
dates = pd.to_datetime(
    pd.read_csv(RAW_CSV, usecols=["EventDate"], dtype=str)["EventDate"].str.split().str[0], format="%m/%d/%Y"
)
if dates.max() != pd.Timestamp(LAST_DAY):
    RAW_CSV.unlink()
    raise WrongReleaseError(
        f"the file's reports run to {dates.max():%d %B %Y}, but the November 2025 release ends on {LAST_DAY}. "
        f"Download it from {NOVEMBER_URL} and run this cell again."
    )
print(
    f"{RAW_CSV.relative_to(REPO_DIR)}: {len(dates):,} reports, {dates.min():%d %b %Y} to {dates.max():%d %b %Y}, columns checked"
)

## 7. Build the data and the indexes

These are the project's own commands (`safereport <command>`, see the README quick start), run in
order. Each one is skipped when its output already exists, so running the notebook again is quick.

| Command | What it builds |
|---|---|
| `build-data` | `reports.parquet`: every report with clean columns, and the fixed 2025 holdout split |
| `fetch-ecfr` | the text of 29 CFR 1910 as of a fixed date, from the public eCFR API (no key) |
| `build-index` | the search indexes over the regulations and OSHA's emphasis programs (`--no-dense`: BM25 only) |
| `train-baseline` | the stage 1 event coder (TF-IDF and logistic regression), trained outside the holdout |
| `check` | lists the model settings and what is built |

In [ ]:
import json
import time


def safereport_cli(*args: str) -> None:
    started = time.perf_counter()
    print(f"$ safereport {' '.join(args)}")
    run(sys.executable, "-m", "safereport.cli", *args, cwd=REPO_DIR)
    print(f"({time.perf_counter() - started:.0f} s)\n")


def index_matches_setting() -> bool:
    manifest = REPO_DIR / "indexes" / "manifest.json"
    return (
        manifest.exists() and bool(json.loads(manifest.read_text()).get("embedding_model")) == USE_EMBEDDINGS
    )


steps = [
    (["build-data"], REPO_DIR / "data" / "processed" / "reports.parquet"),
    (["fetch-ecfr"], REPO_DIR / "data" / "raw" / "ecfr" / "title-29-part-1910.xml"),
    (["build-index"] + ([] if USE_EMBEDDINGS else ["--no-dense"]), None),
    (["train-baseline"], REPO_DIR / "indexes" / "event_baseline.joblib"),
]
for args, output in steps:
    done = index_matches_setting() if output is None else output.exists()
    if done:
        print(f"safereport {args[0]}: already built, skipped")
    else:
        safereport_cli(*args)
safereport_cli("check")

## 8. A first look at the data

Counted from the built file, not typed in: how many reports there are, how many OSHA inspected, and
how the reports are split. The **holdout** is 1,000 reports of 2025 kept aside for the final
evaluation; no rule or threshold is tuned on them.

In [ ]:
from IPython.display import Markdown, display
import pandas as pd


reports = pd.read_parquet(REPO_DIR / "data" / "processed" / "reports.parquet")
inspected = reports["inspected"].astype(bool)
display(
    Markdown(
        f"**{len(reports):,} reports**, {reports['event_date'].min()} to {reports['event_date'].max()}. "
        f"OSHA inspected {inspected.sum():,} of them ({inspected.mean():.1%})."
    )
)
display(reports.groupby("split").size().rename("reports").to_frame())
by_year = reports.groupby("year").agg(reports=("id", "size"), inspected=("inspected", "mean"))
by_year["inspected"] = (by_year["inspected"] * 100).round(1).astype(str) + "%"
display(by_year.T)

## 9. Start the crew, offline

`load_services` loads everything the agents use: the reports, the indexes, the eCFR text, the
stage 1 coder and the model **router**. Without `live=True`, every model behind the router is the
project's `OfflineBackend`: fixed rules that return what a model would, so the whole flow runs
with no key and no cost. The cell stops if any backend is not offline.

**Read the traces below with this in mind.** The router still chooses a model *slot* for each
step (the trace says `claude` or `gpt_nano`, the model it would call live), but the answer
comes from the offline rules. Their outputs show how the agents work together; they are not model
results, and the project never reports them as such. Part 17 covers the real router results.

In [ ]:
from langgraph.types import Command

from safereport.graph import build_graph, load_services, run_config, sqlite_checkpointer


services = load_services(mode="lowest_cost", dense=USE_EMBEDDINGS, live=False)
backends = {name: type(backend).__name__ for name, backend in services.router.backends.items()}
assert set(backends.values()) == {"OfflineBackend"}, f"a backend is not offline: {backends}"
print("Model slots and who answers them:", backends)

week_start, week_end = services.week
print(
    f"Replayed week: {week_start:%A %d %B %Y} to {week_end:%A %d %B %Y}; {services.capacity:g} inspector days by default"
)

graph = build_graph(services, sqlite_checkpointer())  # the checkpointer lets a paused run resume
config = run_config()

## 10. The request, and the Planner's question

The request is the one in the README quick start, in plain words. It names no state, so the
**Planner** does not guess one: the graph **pauses** (a LangGraph *interrupt*) and asks. The run is
saved by the checkpointer, so it can resume later with the answer, even after a restart.

In [ ]:
REQUEST = "Plan this week's inspections with 40 inspector days."
handoffs: list[dict] = []  # every step of the run: which node ran and what it passed on


def stream(value) -> object:
    """Run the graph until it pauses or ends, recording each node's update."""
    paused = None
    for update in graph.stream(value, config, stream_mode="updates"):
        for node, change in update.items():
            if node == "__interrupt__":
                paused = change[0].value
                continue
            change = change or {}
            trace = change.get("trace", [])
            handoffs.append(
                {
                    "step": len(handoffs) + 1,
                    "node": node,
                    "agents": ", ".join(dict.fromkeys(t["agent"] for t in trace)),
                    "passes on": ", ".join(k for k in change if k != "trace") or "(routing only)",
                    "first trace line": trace[0]["summary"] if trace else "",
                }
            )
    return paused


question = stream({"request": REQUEST})
state = graph.get_state(config).values
print("Planner's tasks:", state["tasks"])
print("Parameters so far:", state["params"])
print("Missing:", question["missing"], "->", question["questions"])

## 11. Answer, and let the crew work until it needs a person

We answer *Texas*. The run resumes and goes through stage 1, the parallel stage 2 branches, the
Optimizer, the Writer and the Decision critic (with any send-backs), and pauses again at the
**approval** step: the plan is only a proposal until a person approves it.

The table is the run's own record of the **handoffs**: each node in the order it ran, the agents
it speaks for, and the parts of the shared state it passed on to the next node.

In [ ]:
approval_request = stream(Command(resume={"state": "Texas"}))
state = graph.get_state(config).values

pd.set_option("display.max_colwidth", 90)
handoff_table = pd.DataFrame(handoffs)
display(handoff_table)
print(
    f"{len(handoff_table)} steps; {(handoff_table['node'] == 'branch').sum()} stage 2 branches ran in parallel."
)
print("The run is paused at:", approval_request["question"])

## 12. Stage 1: every report, coded without a model

Stage 1 is the fast path. For each report of the week it codes the event with the trained
classifier, checks the Category 1 rules, matches the employer against past reports, and searches
the regulations (BM25, plus embeddings when `USE_EMBEDDINGS = True`). Its result per report decides
whether that report needs a stage 2 branch: it does when the coder is unsure, when the two best
regulation sections score too close together, when the injury is severe, or when the report is
near the cut of the plan (a Category 1 report only when the coder is unsure).

In [ ]:
def stage1_row(report_id: str, item: dict) -> dict:
    intake = item["intake"]
    return {
        "report": report_id,
        "employer": item["report"]["employer"][:40],
        "hazard (Intake)": intake["hazard_type"],
        "event code": intake["event_code"],
        "confidence": round(intake["confidence"], 2),
        "Category 1": "; ".join(f"{c['criterion']} ({c['evidence']})" for c in item["category_one"]) or "-",
        "standard (stage 1 search)": item["pick"].get("primary", ""),
        "stage 2 branch": report_id in state["stage2"],
    }


stage1 = pd.DataFrame([stage1_row(rid, item) for rid, item in state["stage1"].items()])
display(
    Markdown(
        f"**{len(stage1)} Texas reports** in the week. {stage1['Category 1'].ne('-').sum()} meet a Category 1 "
        f"criterion; {stage1['stage 2 branch'].sum()} were sent to a stage 2 branch, {(~stage1['stage 2 branch']).sum()} "
        "were clear enough for stage 1 alone."
    )
)
display(stage1)

**How this count compares with B4.** The B4 evaluation (`eval/results/batch_savings.json`) measured
the same Texas week and counted the same thing: the reports the fan-out sends to a stage 2 branch.
Its count can differ from the one above because it depends on the regulation search: B4 ran with
the search named in the file's `method.retrieval`, and this notebook uses the one set by
`USE_EMBEDDINGS`. A report goes to stage 2 when the scores of its two best sections are closer
than the router's `min_retrieval_margin`, and the two searches score sections differently. The
cell reads B4's numbers from the file and lists the reports on which the two runs differ, with
the inputs the fan-out read for them in this run.

In [ ]:
b4 = json.loads((REPO_DIR / "eval" / "results" / "batch_savings.json").read_text())
assert (b4["week"]["start"], b4["week"]["end"]) == (str(week_start), str(week_end)), "not the same week"
b4_branches = set(state["stage1"]) - set(b4["reports_skipped_by_two_stages"])
here = set(state["stage2"])
router_settings = services.settings.router
print(
    f"B4 ({b4['method']['retrieval']} search): {b4['two_stages']['reports_through_stage2']} of {b4['two_stages']['reports']} reports through stage 2"
)
print(f"This run ({'hybrid' if USE_EMBEDDINGS else 'bm25'} search): {len(here)} of {len(state['stage1'])}")
print(
    f"Thresholds: min_confidence {router_settings.min_confidence}, min_retrieval_margin {router_settings.min_retrieval_margin}"
)
differ = sorted(here ^ b4_branches)
if differ:
    display(
        pd.DataFrame(
            [
                {
                    "report": rid,
                    "stage 2 here": rid in here,
                    "stage 2 in B4": rid in b4_branches,
                    "coder confidence": round(state["stage1"][rid]["intake"]["confidence"], 3),
                    "retrieval margin here": round(state["stage1"][rid]["margin"], 4),
                }
                for rid in differ
            ]
        )
    )

## 13. Stage 2: one report through Intake, Standards and Analyst

One stage 2 report, chosen as the one with the highest stage 1 priority, shown in full. Each
branch hands the Optimizer three things:

* **Intake**: what happened, the event code with a confidence, and the facts the story leaves
  out (they become questions in the RRI letter);
* **Standards**: the cited section and paragraph, with the official text, and the other
  candidates it weighed;
* **Analyst**: the employer's past reports, possible repeats, how common the hazard is in this
  industry, and similar past injuries.

Here the citation comes from the offline rules and, with `USE_EMBEDDINGS = False`, a keyword search
alone, so read it as an example of what is passed on, not of how accurate the search is. A wrong
section is what the Decision critic is there to catch (part 15); the measured accuracy is in parts
17 and 18.

In [ ]:
chosen = max(state["stage2"], key=lambda rid: state["stage1"][rid]["priority"])
branch_result = state["stage2"][chosen]
report, intake, pick = branch_result["report"], branch_result["intake"], branch_result["pick"]

display(
    Markdown(
        f"**Report {chosen}**, {report['employer']}, {report['city'].title()}, NAICS {report['naics']}\n\n> {report['narrative']}"
    )
)
display(Markdown("**Intake**"))
display(
    pd.Series(
        {
            "activity": intake["activity"],
            "hazard": intake["hazard_type"],
            "machine or hazard": intake["machine_or_hazard"],
            "injury": intake["injury"].strip(),
            "event code": f"{intake['event_code']} (confidence {intake['confidence']:.2f})",
            "facts the story leaves out": "; ".join(intake["missing_facts"]) or "-",
        }
    ).to_frame("value")
)
display(Markdown("**Standards**"))
display(
    pd.Series(
        {
            "cited": f"{pick['primary']} {pick.get('secondary') or ''}".strip(),
            "paragraph": pick.get("paragraph", ""),
            "paragraph text": (pick.get("paragraph_text") or "")[:300],
            "other candidates": ", ".join(branch_result["candidates"][1:]),
            "emphasis programs": ", ".join(branch_result["emphasis"]) or "-",
        }
    ).to_frame("value")
)
display(Markdown("**Analyst**"))
display(
    pd.Series(
        {
            "employer's past reports (same name)": branch_result["history_same"],
            "possible matches to confirm": branch_result["history_possible"],
            "repeats within a year": len(branch_result["repeats"]),
            "share of this hazard in the industry": f"{branch_result['industry_share']:.1%}",
        }
    ).to_frame("value")
)
similar = pd.DataFrame(branch_result["similar"])
if not similar.empty:
    display(similar[["report_id", "event_date", "state", "narrative"]])

## 14. Optimizer: Category 1 first, then the inspector days

Every Category 1 report is inspected. The integer program then spends the inspector days left on
the reports with the highest priority per day. Each row carries the reason a person will read.
If the days are too few even for Category 1, a relaxed model says how many more are needed
(`shortfall`).

In [ ]:
triage = state["findings"]["triage"]
plan = triage["plan"]
rows = pd.DataFrame(triage["rows"])
display(
    Markdown(
        f"**{(rows['decision'] == 'inspection').sum()} inspections and {(rows['decision'] == 'rri_letter').sum()} RRI letters**, "
        f"using {plan['days_used']:g} of {triage['capacity']:g} inspector days; shortfall {plan['shortfall']:g} days."
    )
)
display(rows[["report_id", "employer", "decision", "priority", "days", "standard", "reason"]])

## 15. Writer and Decision critic

The **Writer** never types a number. Its drafts hold placeholders such as `{{inspections}}`, and
each placeholder is filled from a **ledger** of facts that a tool recorded, with the tool's name
as the source. Below: the placeholders of the week's summary with their sources, and one RRI
letter as it would be sent.

The **Decision critic** then checks each citation against the eCFR text and OSHA's own rules,
each number, and the plan rules. A problem it can fix goes back to the Writer or the Optimizer (at
most twice); one it cannot goes to a person, listed under the answer.

In [ ]:
from safereport.ledger import FactLedger


def render(draft: dict) -> str:
    ledger = FactLedger()
    for key, (value, source, same_months) in draft["facts"].items():
        ledger.add(key, value, source, same_months=same_months)
    return ledger.render(draft["template"])


summary_draft = state["drafts"]["triage:summary"]
display(
    pd.DataFrame(
        [{"placeholder": k, "value": v[0], "recorded by": v[1]} for k, v in summary_draft["facts"].items()]
    )
)
letters = sorted(k for k in state["drafts"] if k.startswith("triage:letter_"))
print(f"{len(letters)} RRI letters drafted. The first one:\n")
if letters:
    print(render(state["drafts"][letters[0]]))

In [ ]:
critic_lines = [t for t in state["trace"] if t["agent"] == "Decision critic"]
print(
    f"Decision critic: {len(critic_lines)} checks recorded; Writer send-backs: {state.get('writer_retries', 0)}"
)
for line in critic_lines:
    print(" -", line["summary"])
issues = pd.DataFrame(state.get("issues", []))
if issues.empty:
    print("No open issue for a person.")
else:
    display(Markdown("**Left for a person to check** (route `person`):"))
    display(issues[["report", "rule", "message", "suggestion"]])

## 16. A person decides

The run is paused at the approval step. First the Area Director **asks for a change**: one report
planned for a letter is moved to an inspection. The graph resumes at the Optimizer with that rule,
re-plans within the same days, and the Writer and critic run again. Then the Area Director
**approves**, and the decision is logged with who made it.

In [ ]:
letter_rows = rows[rows["decision"] == "rri_letter"]
if letter_rows.empty:
    print("Every report is already planned for an inspection; no change to ask for.")
else:
    moved = letter_rows.iloc[0]["report_id"]
    before = dict(plan["decisions"])
    stream(
        Command(
            resume={
                "action": "changes",
                "by": "Area Director (notebook)",
                "force_inspection": [moved],
                "notes": {moved: "notebook demo: inspect this one"},
            }
        )
    )
    state = graph.get_state(config).values
    after = state["findings"]["triage"]["plan"]["decisions"]
    changed = {rid: f"{before[rid]} -> {after[rid]}" for rid in after if before.get(rid) != after[rid]}
    print(f"Moved {moved} to an inspection. Rows that changed: {changed}")
    print(f"Inspector days used: {state['findings']['triage']['plan']['days_used']:g}")

stream(Command(resume={"action": "approve", "by": "Area Director (notebook)"}))
state = graph.get_state(config).values
display(
    pd.DataFrame(
        [{k: v for k, v in entry.items() if k in ("at", "by", "action")} for entry in state["decision_log"]]
    )
)
display(Markdown(state["answer"]))

### What each agent contributed to this week

Counted from the run's trace and state above, so the numbers are this run's, not typed in.

In [ ]:
trace = pd.DataFrame(state["trace"])
final_plan = state["findings"]["triage"]["plan"]["decisions"]
contribution = {
    "Planner": f"{len(state['tasks'])} task(s); asked for {question['missing']}",
    "Intake": f"{len(state['stage1'])} reports coded in stage 1, {len(state['stage2'])} read again in a branch",
    "Standards": f"{len(state['stage2'])} citations with section and paragraph",
    "Analyst": f"{sum(len(b['similar']) for b in state['stage2'].values())} similar past injuries, "
    f"{sum(len(b['repeats']) for b in state['stage2'].values())} repeats found",
    "Optimizer": f"{sum(v == 'inspection' for v in final_plan.values())} inspections, "
    f"{sum(v == 'rri_letter' for v in final_plan.values())} letters",
    "Writer": f"{len(state['drafts'])} drafts, every number from the ledger",
    "Decision critic": f"{len(state.get('issues', []))} issue(s) left for a person",
}
per_agent = trace.groupby("agent").size().rename("trace lines").to_frame()
per_agent["model slots used"] = trace.groupby("agent")["model"].apply(
    lambda m: ", ".join(sorted(set(m) - {""})) or "none"
)
per_agent["contribution"] = pd.Series(contribution)
display(per_agent.reindex([*contribution, *(a for a in per_agent.index if a not in contribution)]).fillna(""))

# Names the later sections can use: services, graph, config, state, handoff_table, stage1, rows.

## 17. The model router (Khoa)

*To be written by Khoa:* the accuracy per model and step (the B7 numbers, #8), read from
`eval/results/`, and why each step goes to the model it goes to.

## 18. The evaluation numbers (Khoa)

*To be written by Khoa:* the measured results, read from the files in `eval/results/` of the pinned
copy (`REPO_DIR / "eval" / "results"`), not typed in.

## 19. Edge cases (Khoa)

*To be written by Khoa:* a few cases the system handles, for example a very short narrative and an
invented citation caught by the critic.

## 20. Summary

* The notebook installed SafeReport Crew at a pinned commit, loaded OSHA's public file and built the
  data and indexes with the project's own commands.
* It ran one real week of Texas reports through the seven agents offline: the Planner asked for the
  missing state, stage 1 coded every report without a model, stage 2 branches read the unclear ones,
  the Optimizer planned the inspector days, the Writer drafted from recorded facts only, the critic
  checked the work, and a person asked for a change and approved.
* No API key was read and no model was called; the offline rules stand in for the models.

More: the [README](https://github.com/lequo-neu/safereport-crew#readme) (results and limits), and
`safereport ui` for the full app.